# 01 — Competition Analysis

**Purpose:** Establish a verified, written-down understanding of the task before any modeling begins.

This notebook should orchestrate reading of the official competition pages / downloaded files, not contain modeling logic.

Sections are filled in only from the official Overview, Data, and Rules pages (logged in), or from the actual downloaded competition files. Do not carry over unverified claims from public notebooks, other repos, or news coverage — flag those separately as unverified if mentioned at all.

## 1. Task Definition

**Confirmed (official Overview → Description / Evaluation, 27 Aug 2026):**

- Multimodal task: predict twelve clinically important knee abnormalities per study, using both the MRI images and the study's original radiology report text.
- Granularity: one prediction row per `StudyInstanceUID`.
- Twelve targets (verbatim, submission-file order):
  1. ACL
  2. MCL
  3. Medial Meniscus
  4. Lateral Meniscus
  5. Medial OA
  6. Lateral OA
  7. PF OA
  8. Effusion
  9. Synovitis
  10. Baker's
  11. Contusion
  12. Fracture
- Multilabel per study (a study predicts a confidence score for all 12 targets independently).

**Open sub-question:** three targets (Medial OA, Lateral OA, PF OA) are laterality-specific by name; the other nine are not. Need to confirm from the data description whether each study represents a single knee, or whether non-OA findings are pooled across both knees within a study.

## 2. Evaluation Metric

**Confirmed (official Overview → Evaluation, 27 Aug 2026):**

- Macro-averaged ROC-AUC across the twelve targets.
- `Final Score = (1/12) * sum(AUC_i for i in the 12 targets)`
- Equal weight per target regardless of class balance.

**Still open:**

- Efficiency-prize evaluation criteria — separate track, own scoring, not yet reviewed.

## 3. Submission Format

**Confirmed (official Overview → Evaluation, 27 Aug 2026):**

CSV with header row, one row per `StudyInstanceUID`, one confidence-score column (range 0–1) per target, in this column order:

StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
<uid_1>,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
<uid_2>,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5

## 4. Dataset Structure

**Confirmed (official Data description, 27 Aug 2026):**

- `train.csv` — one row per study: `StudyInstanceUID`, `Report` (free-text, multiple languages), 12 binary labels.
- `train_series.csv` — one row per series: `StudyInstanceUID`, `SeriesInstanceUID`, `Fluid_Sensitive` (0/1), `Fat_Suppression` (0/1, not always equal to `Fluid_Sensitive`), `Anatomical_Plane` (Sagittal/Coronal/Axial).
- `train_series/<StudyInstanceUID>/<SeriesInstanceUID>/<SOPInstanceUID>.dcm` — one slice per file. Series: 20–45 slices typical, median 30, long tail to a few hundred.
- `test.csv` — ~1,300 studies at scoring time (example file has 3). `StudyInstanceUID` only — **no `Report` field.**
- `test_series.csv` / `test_series/` — same schema as train, swapped at scoring time.
- `sample_submission.csv` — all label columns = 0.5.
- Distribution notice: abnormality prevalence not guaranteed consistent across train / public leaderboard / final evaluation.
- DICOM notes: variable intensity/orientation/resolution; mixed transfer syntaxes (uncompressed Explicit VR LE, JPEG Lossless, JPEG 2000, Implicit VR LE) — reader needs `pylibjpeg`/`gdcm`, not just plain `pydicom`; metadata stripped to an allowlisted set of 86 tags.

**Critical architectural constraint:** report text is available at train time but explicitly withheld at test time. Any report-derived signal (labels, weak supervision, auxiliary training targets) must be used only during training — the deployed/inference model can only see images.

**Still open:** whether each study represents a single knee. No laterality field exists in `train_series.csv`, which is circumstantial evidence for "yes," but this needs to be confirmed by inspecting actual studies (e.g. checking whether a study ever contains series for both left and right knees).

## 5. Labels and Ground Truth

**Confirmed (official Data description, 27 Aug 2026):**

- Only a small subset of training studies carry direct per-condition labels.
- The remaining studies have only the original radiology report text, from which labels may be derived.
- This is confirmed as a weak-supervision problem, not plain fully-labeled multilabel classification.

**Still to measure directly from `train.csv` once downloaded:**

- Exact count/proportion of studies with direct labels vs. report-only.
- Class balance per target, for both the labeled subset and any derived labels.
- Report language distribution.

**Unverified secondary-source claim (do not rely on until measured ourselves):** other participants have reported the labeled subset is very small relative to the full training set (e.g. on the order of ~1% in one public repo), and that naively derived report labels agree with expert labels only around ~82% of the time. The general shape of this claim is now consistent with the official data description, but the specific numbers must be measured from our own copy of `train.csv`, not assumed.

## 6. Rules and Constraints

**Confirmed (official Rules / Code Requirements, 27 Aug 2026):**

- Timeline: start Jul 30, 2026 · entry/team-merge deadline Oct 15, 2026 · final submission Oct 22, 2026 · winners' requirement deadline Nov 5, 2026. All 11:59 PM UTC.
- Code competition: submissions run as Notebooks.
  - ≤9 hours runtime (CPU or GPU).
  - Internet access disabled at submission time.
  - Freely & publicly available external data and pretrained models are allowed.
  - Output must be named `submission.csv`.
- Winners' Obligations (beyond standard Kaggle terms): short video of approach, public code + weights on the competition forum, final model publicly available for open distribution/validation.

**Still open:**

- Data security / report-text handling constraints — not found explicitly in the Rules text reviewed so far. Verify explicitly before using any hosted LLM API on report text (only relevant during training, since report text isn't available at inference anyway).

## 7. Timeline

**Confirmed (official Timeline, 27 Aug 2026):**

| Milestone | Date |
|---|---|
| Start | Jul 30, 2026 |
| Entry deadline | Oct 15, 2026 |
| Team merger deadline | Oct 15, 2026 |
| Final submission deadline | Oct 22, 2026 |
| Winners' requirement deadline | Nov 5, 2026 |

All 11:59 PM UTC on the stated day; organizers may update the timeline.

## 7b. Prizes and Efficiency Track

**Confirmed (official Prizes / Efficiency Prize Evaluation, 27 Aug 2026):**

- Main Leaderboard: 10 prizes, $5,000–$9,000.
- Efficiency Track: 3 prizes, $5,000–$7,000. A submission can win both tracks.
- Efficiency eligibility: must be a selected/auto-selected submission, and must beat `sample_submission.csv` on the Private Leaderboard.
- Efficiency score (lower is better):

  `Efficiency = AUC / (Benchmark − max AUC) + RuntimeSeconds / 32400`

  - `AUC` — submission's score on the main metric.
  - `Benchmark` — `sample_submission.csv`'s AUC.
  - `max AUC` — best Private Leaderboard AUC across all submissions.
  - `RuntimeSeconds` — this submission's own evaluation runtime.
  - `32400` — 9-hour runtime cap, in seconds (i.e. runtime normalized against the max allowed).

**Note to double check once real numbers exist:** since `max AUC > Benchmark` is expected, `Benchmark − max AUC` is presumably negative, which affects how the AUC term's sign/scale behaves — worth sanity-checking against the daily-updated public Efficiency Leaderboard once available, rather than assuming the formula's intuitive direction from the LaTeX alone.

## 8. Open Questions / Risks

- [x] Evaluation metric — confirmed.
- [x] Task definition and submission format — confirmed.
- [x] Dataset file structure / DICOM hierarchy — confirmed.
- [x] Label source and coverage — confirmed as weak-supervision shaped; exact numbers still to measure.
- [x] Report-text availability at inference — confirmed: training-time only.
- [x] Rules, deadline, code-competition constraints — confirmed.
- [x] Efficiency-prize evaluation criteria — confirmed.
- [ ] Whether a study = one knee (circumstantial evidence, not explicit)
- [ ] Patient/site leakage risk and official split structure
- [ ] Data-security constraints on report text (if any) — not yet located in Rules text

Mirror significant items back into the repo `README.md` when resolved.